In [ ]:
%load_ext autoreload
%autoreload 2
# Import required libraries
import sys
sys.path.append("../")
import os
import numpy as np
import utm
import io
import shutil
import h5py
import json
import matplotlib.pyplot as plt
from PIL import Image
from typing import List
import cv2 
from pyproj import CRS, Transformer
import json
import pandas as pd

# Load customized utility functions
from utils.transformations import rotation_matrix_from_angles, get_yaw_pitch_roll, filter_above_ground, compute_heading
from utils.plot import plot_lidar_camera_oxts
from utils.process import memory_usage

# Load ZOD DevKit
from zod import ZodFrames, ZodSequences, ZodDrives
import zod.constants as constants
from zod.constants import Camera, Lidar, Anonymization, AnnotationProject
from zod.data_classes.ego_motion import OXTS_TIMESTAMP_OFFSET, interpolate_transforms
from zod.data_classes import LidarData
from zod.utils.geometry import transform_points
from zod.visualization.lidar_on_image import get_3d_transform_camera_lidar
from zod.constants import Camera, Lidar
from zod.data_classes.metadata import SequenceMetadata

# Load aerial image download functions
from Map_Downloading.scripts.sweden import sweden_exact_position_image
from Map_Downloading.scripts.sweden import RESOLUTION as sweden_resolution
from Map_Downloading.scripts.france import france_exact_position_image
from Map_Downloading.scripts.france import TRUE_RESOLUTION as france_resolution


# Set matplotlib settings
%matplotlib inline

# NOTE! Set the path to dataset and choose a version
lidar_dataset_root = "/work/vita/datasets/zod"  # your local path to zod "/mnt/data/Work/datasets/zod"
aerial_img_dataset_root = "/work/vita/datasets/zod_crossview_processed_100m"  # your local path to aerial images
version = "full"  # "mini" or "full"



# Load and display sensor frames
sensor_frames = plt.imread('./sensor_frames.png')
plt.figure(figsize=(20, 10))
plt.imshow(sensor_frames)
plt.axis('off')
plt.show()


In [ ]:
def approximate_meridian_convergence(lat_deg, lon_deg, central_meridian_deg):
    """
    Approximates the meridian convergence (in degrees) given a point's latitude and longitude,
    and the central meridian of the map projection.

    Args:
        lat_deg (float): Latitude in degrees.
        lon_deg (float): Longitude in degrees.
        central_meridian_deg (float): Central meridian longitude in degrees.

    Returns:
        float: Meridian convergence in degrees.
    """
    lat_rad = np.radians(lat_deg)
    lon_rad = np.radians(lon_deg)
    cm_rad = np.radians(central_meridian_deg)
    gamma_rad = np.arctan(np.tan(lon_rad - cm_rad) * np.sin(lat_rad))
    return np.degrees(gamma_rad)

In [ ]:
# initialize ZodDrives
zod_drives = ZodDrives(dataset_root=lidar_dataset_root, version=version)


# get default training and validation splits
training_drives = zod_drives.get_split(constants.TRAIN)
validation_drives = zod_drives.get_split(constants.VAL)

print(f"Number of training drives: {len(training_drives)}")
print(f"Number of validation drives: {len(validation_drives)}")

all_drive_num = list(training_drives) + list(validation_drives)
print(f"Number of total drives: {len(all_drive_num)}")

drives_per_country = {}

for i in range(len(all_drive_num)):
    zod_drive = zod_drives[all_drive_num[i]]
    # zod_drive._metadata = SequenceMetadata("/work/vita/datasets/zod/drives/000000/metadata.json")
    metadata = zod_drive.metadata
    
    if metadata.country_code in drives_per_country:
        drives_per_country[metadata.country_code].append(all_drive_num[i])
    else:
        drives_per_country[metadata.country_code] = [all_drive_num[i]]

for country in drives_per_country.keys():
    print(country, len(drives_per_country[country]))

### Use previous and next oxts measure (global positioning) to get current pose (recommended)

In [ ]:
size = 100 # side length (m) of the requested aerial image


# List of drives to process
drive_list = [str(i).zfill(6) for i in range(28)]

# drive_list = ['000021']  # Modify as needed

for drive_idx in drive_list:
    print('drive_idx', drive_idx)
    drive = zod_drives[drive_idx]
    
    ######################################################################
    # MF : filepath to corresponding aerial image in cross view dataset
    temp_filepath_to_aerial_img = drive.info.camera_frames['front_blur'][i].filepath
    temp_filepath_to_aerial_img = temp_filepath_to_aerial_img.replace("zod", "zod_crossview_processed_100m" ).replace("drives","drives/"+drive.metadata.country_code).replace("camera_front_blur", "aerial").replace(".jpg", ".png")
    # print(temp_filepath_to_aerial_img)
    
    tmp = aerial_img_dataset_root + "/drives/" + drive.metadata.country_code + "/labels.json" 

    with open(tmp, 'r') as f:
        aer_img_data_temp = json.load(f)
        
    aerial_img_data_df = pd.DataFrame(aer_img_data_temp)
    
    if drive.metadata.country_code != 'SE':
        continue
    if drive_idx not in aerial_img_data_df.drive_idx.unique(): 
        continue
    ######################################################################

    # Define Paths
    drive_path = os.path.join(lidar_dataset_root, 'drives', drive_idx)
    aerial_path = os.path.join(aerial_img_dataset_root, "aerial_img")
    ground_path = os.path.join(aerial_img_dataset_root, "ground_img")

    calibrations = drive.calibration
    T_lidar2oxts = calibrations.lidars[Lidar.VELODYNE].extrinsics.transform

    num_frames = len(drive.info.camera_frames['front_blur'])
    filename = os.path.join(drive_path, "oxts.hdf5")

    # Read HDF5 Data Once
    with h5py.File(filename, "r") as f:
        def recursively_print(name, obj):
            print(name)
        # f.visititems(recursively_print)
        # print('GNSSAntenna/accuracyHeadingAntennas', f['GNSSAntenna/accuracyHeadingAntennas'][()])
        # print('poseSource', f['poseSource'][()])

        orientationMode = f['orientationMode'][()]
        earth_frame_bytes = f['earthFrame'][()]
        earth_frame = b''.join(earth_frame_bytes).decode('utf-8') 

        INSToHostRotation = f['INSToHostRotation'][()]

        datumEllipsoid_bytes = f['datumEllipsoid'][()]
        datumEllipsoid = b''.join(datumEllipsoid_bytes).decode('utf-8') 
        
        lat = f['posLat'][()]
        lon = f['posLon'][()]
        alt = f['posAlt'][()]
        print(f['heading'][()] )
        yaw = 90 - f['heading'][()] 
        
        pitch = f['pitch'][()]
        roll = f['roll'][()]
        timestamp = OXTS_TIMESTAMP_OFFSET + f['timestamp'][()] + f['leapSeconds'][()][0]
        
    # print('earth_frame', earth_frame)
    # print('datumEllipsoid', datumEllipsoid)
    # print('INSToHostRotation', (INSToHostRotation))
    # print('orientationMode', orientationMode)
    
    heading_difference = []
    T_prev = np.eye(4)


    # for i in range(0, num_frames):
    for i in range(0, 1):
        if i % 100 == 0:
            print(f"Frame {i}/{num_frames}, Memory Usage: {memory_usage():.2f} MB")
            

        current_timestamp = drive.info.camera_frames['front_blur'][i].time.timestamp()
        
        # Use np.searchsorted for efficient timestamp lookup
        oxts_idx = np.searchsorted(timestamp, current_timestamp, side='right')
        print('current_timestamp', current_timestamp, 'oxts_idx', oxts_idx)


        if oxts_idx == 0 or oxts_idx >= len(timestamp):
            continue  # Skip if no valid index

        # Extract previous and next OXTS readings
        lat_prev, lon_prev, alt_prev = lat[oxts_idx - 1], lon[oxts_idx - 1], alt[oxts_idx - 1]
        yaw_prev, pitch_prev, roll_prev = yaw[oxts_idx - 1], pitch[oxts_idx - 1], roll[oxts_idx - 1]
        easting_prev, northing_prev, zone_number_prev, zone_letter_prev = utm.from_latlon(lat_prev, lon_prev)

        lat_next, lon_next, alt_next = lat[oxts_idx], lon[oxts_idx], alt[oxts_idx]
        yaw_next, pitch_next, roll_next = yaw[oxts_idx], pitch[oxts_idx], roll[oxts_idx]
        easting_next, northing_next, zone_number_next, zone_letter_next = utm.from_latlon(lat_next, lon_next)
        
        lat_aerial, lon_aerial = aerial_img_data_df.loc[cross_view_idx].aerial_latlon
        easting_aerial, northing_aerial, zone_number_aerial, zone_letter_aerial = utm.from_latlon(lat_aerial, lon_aerial)


        gamma_prev = 0
        gamma_next = 0
        if drive.metadata.country_code == 'SE':
            gamma_prev = approximate_meridian_convergence(lat_prev, lon_prev, 15)
            gamma_next = approximate_meridian_convergence(lat_next, lon_next, 15)
        
        yaw_prev += gamma_prev # heading should subtract gamma, hence yaw should be adding 
        yaw_next += gamma_next 


        # Compute previous and next transformation matrices
        T_previous = np.eye(4)
        T_previous[:3, :3] = rotation_matrix_from_angles(
            np.radians(roll_prev), np.radians(pitch_prev), np.radians(yaw_prev), order="ZYX"
        )
        T_previous[:3, 3] = [easting_prev, northing_prev, alt_prev]

        T_next = np.eye(4)
        T_next[:3, :3] = rotation_matrix_from_angles(
            np.radians(roll_next), np.radians(pitch_next), np.radians(yaw_next), order="ZYX"
        )
        T_next[:3, 3] = [easting_next, northing_next, alt_next]

        
        
        # Interpolate transformation
        interp_factor = (current_timestamp - timestamp[oxts_idx - 1]) / (timestamp[oxts_idx] - timestamp[oxts_idx - 1])
        T_current = interpolate_transforms(T_previous, T_next, interp_factor)
        yaw_oxts, _, _ = get_yaw_pitch_roll(T_current)
        heading_oxts = 90 - np.degrees(yaw_oxts)
        
        
    
        # Process Lidar Data
        pcd = drive.get_compensated_lidar(drive.info.camera_frames['front_blur'][i].time).points
        pcd_utm = transform_points(pcd, T_current @ T_lidar2oxts) 
        pcd_utm = filter_above_ground(pcd_utm, ground_threshold=1)

        # Compute heading difference
        heading_move = compute_heading(T_prev[0, 3], T_prev[1, 3], T_current[0, 3], T_current[1, 3])

        if i > 0:
            difference = (heading_move - heading_oxts + 180) % 360 - 180
            # print('difference', difference)
            heading_difference.append(difference)

        T_prev = T_current.copy()


        # Extract aerial image metadata
        aerial_img_rel_path = temp_filepath_to_aerial_img.replace(aerial_img_dataset_root+"/", "")
        cross_view_idx = aerial_img_data_df.loc[(aerial_img_data_df.aerial_image == aerial_img_rel_path)].index.item()
        aer_img_metadata = aerial_img_data_df.loc[cross_view_idx]
        
        aerial_lat, aerial_lon = aer_img_metadata['aerial_latlon']
        # ground_lat, ground_lon = aer_img_metadata['ground_latlon']
        aerial_easting, aerial_northing, _, _ = utm.from_latlon(aerial_lat, aerial_lon)
        # ground_easting, ground_northing, _, _ = utm.from_latlon(ground_lat, ground_lon)
        dx_offset = aer_img_metadata['dx']
        dy_offset = aer_img_metadata['dy']
        aerial_heading = aer_img_metadata['heading']
        gamma_aerial = approximate_meridian_convergence(aerial_lat, aerial_lon, 15)
        
        T_aerial = np.eye(4)
        T_aerial[:3, :3] = rotation_matrix_from_angles(
            np.radians(0), np.radians(0), np.radians(gamma_aerial), order="ZYX"
        )
        T_aerial[:3, 3] = [easting_aerial, northing_aerial, 0]
        
        temp_yaw,_,_ = get_yaw_pitch_roll( T_aerial )
        print(f"heading_oxts:{90-np.degrees(temp_yaw)}, yaw : {np.degrees(temp_yaw)}, gamma : {gamma_aerial}")

        
        # Load Aerial Image and Resolution
        aerial_img = cv2.imread(temp_filepath_to_aerial_img)
        resolution = aer_img_metadata.resolution

        # Plot BEV (Bird’s Eye View) with LiDAR points
        bev_image = np.array(aerial_img)
        H, W = bev_image.shape[:2]
        bev_center = (aerial_easting, aerial_northing) # (T_current[0, 3], T_current[1, 3])
        
        ##############################################################################################################
        # Transform lidar from UTM to aerial image coordinates
        # Step 1: Translate relative to aerial image center
        pcd_relative = pcd_utm[:, :2] - np.array([aerial_easting, aerial_northing])

        # Step 2: Rotate by aerial heading
        # Aerial heading is the orientation of the aerial image
        theta = np.radians(heading_oxts-aerial_heading)  # negative for world-to-image transformation
        # theta = 0
        rotation_matrix = np.array([
            [np.cos(theta), -np.sin(theta)],
            [np.sin(theta), np.cos(theta)]
        ])
        pcd_rotated = pcd_relative @ rotation_matrix.T

        # Step 3: Convert from meters to pixels
        # pcd_pixels = pcd_rotated / resolution

        # Step 4: Apply image coordinate transformation
        # Image y-axis points down, so flip y
        # x_indices = (pcd_pixels[:, 0] + W / 2 + dx_offset).astype(int)
        # y_indices = (H / 2 - pcd_pixels[:, 1] + dy_offset).astype(int)
        
        # Convert LiDAR points to BEV pixel coordinates
        # x_indices = ((pcd_utm[:, 0] - bev_center[0]) / resolution + W / 2).astype(int)
        # y_indices = H - ((pcd_utm[:, 1] - bev_center [1]) / resolution + H / 2).astype(int)
        x_indices = ((pcd_rotated[:, 0]) / resolution + W / 2).astype(int)
        y_indices = H - ((pcd_rotated[:, 1]) / resolution + H / 2).astype(int)
        x_indices, y_indices = np.clip(x_indices, 0, W - 1), np.clip(y_indices, 0, H - 1)


        # Clip to image bounds
        x_indices = np.clip(x_indices, 0, W - 1)
        y_indices = np.clip(y_indices, 0, H - 1)

        # Transform vehicle heading for visualization
        # Vehicle position relative to aerial center
        vehicle_relative = np.array([T_current[0, 3] - aerial_easting, 
                                    T_current[1, 3] - aerial_northing])
        # vehicle_rotated = vehicle_relative @ rotation_matrix.T
        vehicle_pixel = vehicle_relative / resolution
        # vehicle_x = int(vehicle_pixel[0] + W / 2 + dx_offset)
        # vehicle_y = int(H / 2 - vehicle_pixel[1] + dy_offset)
        vehicle_x = int(W / 2 + vehicle_pixel[0])
        vehicle_y = int(H / 2 - vehicle_pixel[1])
        
        print('vehicle_pix', vehicle_pixel)
        print("dxy_offset",dx_offset, dy_offset)

        # Rotate heading to match aerial image orientation
        heading_in_aerial_frame = aerial_heading # heading_oxts - aerial_heading
        print(heading_oxts, aerial_heading )
        ##############################################################################################################

        plt.figure(figsize=(8, 8))
        plt.imshow(bev_image, origin='upper')
        plt.scatter(x_indices, y_indices, s=0.1, c='purple', alpha=0.3, label="LiDAR Points")
        # plt.quiver(W / 2, H / 2, np.sin(np.radians(heading_move)), np.cos(np.radians(heading_move)), color='cyan', scale=20, label='Movement')
        plt.quiver(vehicle_x, vehicle_y, np.sin(np.radians(heading_in_aerial_frame)), np.cos(np.radians(heading_in_aerial_frame)), color='r', scale=20, label='OXTS Heading')
        plt.legend(loc=2)
        plt.axis('off')
        # plt.savefig(os.path.join(aerial_path, f'frame{i:06}.png'), bbox_inches='tight')
        plt.show()
        plt.close()
    break

    print(f"Drive {drive_idx} Completed. Heading Difference Mean: {np.mean(heading_difference):.2f}, Median: {np.median(heading_difference):.2f}")

In [ ]:
# Convert aerial image center to UTM
aerial_easting, aerial_northing, zone_num, zone_letter = utm.from_latlon(aerial_lat, aerial_lon)

bev_image = np.array(aerial_img)
H, W = bev_image.shape[:2]
bev_center = (aerial_easting, aerial_northing)


# Transform lidar from UTM to aerial image coordinates
# Step 1: Translate relative to aerial image center
pcd_relative = pcd_utm[:, :2] - np.array([aerial_easting, aerial_northing])

# Step 2: Rotate by DIFFERENCE in heading
# If aerial_heading is the image orientation and heading_oxts is vehicle heading
# OR if aerial image is North-up, no rotation needed - just comment out this section:

# Option A: If aerial image is North-up, skip rotation entirely
# pcd_rotated = pcd_relative  # No rotation needed

# Option B: If aerial image is rotated, use heading difference
heading_difference = heading_oxts - aerial_heading
theta = np.radians(-heading_difference)  
rotation_matrix = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta), np.cos(theta)]
])
pcd_rotated = pcd_relative @ rotation_matrix.T

# Step 3: Convert from meters to pixels
pcd_pixels = pcd_rotated / resolution

# Step 4: Convert to image coordinates (flip y-axis)
x_indices = (pcd_pixels[:, 0] + W / 2).astype(int)
y_indices = (H / 2 - pcd_pixels[:, 1]).astype(int)

x_indices = np.clip(x_indices, 0, W - 1)
y_indices = np.clip(y_indices, 0, H - 1)
plt.figure(figsize=(8, 8))
plt.imshow(bev_image, origin='upper')
plt.scatter(x_indices, y_indices, s=0.1, c='purple', alpha=0.3, label="LiDAR Points")
# plt.quiver(W / 2, H / 2, np.sin(np.radians(heading_move)), np.cos(np.radians(heading_move)), color='cyan', scale=20, label='Movement')
plt.quiver(vehicle_x, vehicle_y, np.sin(np.radians(heading_in_aerial_frame)), np.cos(np.radians(heading_in_aerial_frame)), color='r', scale=20, label='OXTS Heading')
plt.legend(loc=2)
plt.axis('off')
# plt.savefig(os.path.join(aerial_path, f'frame{i:06}.png'), bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
heading_oxts-aerial_heading

In [ ]:
aerial_img_dataset_root + "/drives/" + drive.metadata.country_code + "/" + drive_idx + "/aerial/" + drive_idx + "_frame" + f"{i:06}" + ".png"

In [ ]:
temp_filepath_to_aerial_img = drive.info.camera_frames['front_blur'][i].filepath
temp_filepath_to_aerial_img = temp_filepath_to_aerial_img.replace("zod", "zod_crossview_processed_100m" ).replace("drives","drives/"+drive.metadata.country_code).replace("camera_front_blur", "aerial").replace(".jpg", ".png")
print(temp_filepath_to_aerial_img)

In [ ]:
aer_img = cv2.imread(temp_filepath_to_aerial_img)
plt.imshow(aer_img, origin='upper')
plt.scatter(x_indices, y_indices, s=0.1, c='purple', alpha=0.3, label="LiDAR Points")
# plt.quiver(W / 2, H / 2, np.sin(np.radians(heading_move)), np.cos(np.radians(heading_move)), color='cyan', scale=20, label='Movement')
plt.quiver(W / 2, H / 2, np.sin(np.radians(heading_oxts)), np.cos(np.radians(heading_oxts)), color='r', scale=20, label='OXTS')
plt.legend(loc=2)
plt.axis('off')
# plt.savefig(os.path.join(aerial_path, f'frame{i:06}.png'), bbox_inches='tight')
plt.show()
plt.close()
# display cv2 image aer_img


In [ ]:
import json
import pandas as pd
tmp = aerial_img_dataset_root + "/drives/" + drive.metadata.country_code + "/labels.json" 

with open(tmp, 'r') as f:
    data = json.load(f)
    
temp_df = pd.DataFrame(data)

# Example: Print the first entry
print(data[0])
print(data[0]["ground_latlon"])




In [ ]:
aerial_img_rel_path = temp_filepath_to_aerial_img.replace(aerial_img_dataset_root+"/", "")
temp_df.loc[(temp_df.aerial_image == aerial_img_rel_path)].K.to_numpy()

In [ ]:
temp_df
# aerial_img_rel_path


In [ ]:
drive_idx, temp_filepath_to_aerial_img.replace(aerial_img_dataset_root+"/", "")

In [ ]:
T_current @ T_lidar2oxts

In [ ]:
np.array(temp_df.loc[(temp_df.aerial_image == aerial_img_rel_path)].K.item())

In [ ]:
idx = temp_df.loc[(temp_df.aerial_image == aerial_img_rel_path)].index.item()
np.array(temp_df.iloc[idx].K.item())

In [ ]:
np.linalg.inv(temp_df.iloc[idx].K)

In [ ]:
import numpy as np
T_cross_view = np.hstack([
            np.vstack([ np.array([[1,2,3],[2,3,4],[3,4,5]]), np.array([[0,0,0]]) ]),
            np.array([0,0,0,1]) ])

In [ ]:
np.array([[1,2,3],[2,3,4],[3,4,5]])

In [ ]:
aosudb = np.vstack([ np.hstack([ np.array([[1,2,3],[2,3,4],[3,4,5]]), np.array([[0],[0],[0]]) ]), np.array([0,0,0,1]) ])


In [ ]:
aosudb

In [ ]:
T_cross_view @ T_current @ T_lidar2oxts

In [ ]:
T_current @ T_cross_view @ T_lidar2oxts

In [ ]:
T_current @ T_lidar2oxts

In [ ]:
np.linalg.inv(temp_df.iloc[idx].K)
aosudb = np.vstack([ np.hstack([ np.linalg.inv(temp_df.iloc[idx].K), np.array([[0],[0],[0]]) ]), np.array([0,0,0,1]) ])


In [ ]:
T_current @ T_lidar2oxts

In [ ]:
T_aerial

In [ ]:
# Transform lidar from UTM to aerial image coordinates
# Step 1: Translate relative to aerial image center
pcd_relative = pcd_utm[:, :2] - np.array([aerial_easting, aerial_northing])

# Step 2: Rotate by aerial heading
# Aerial heading is the orientation of the aerial image
theta = np.radians(-aerial_heading)  # negative for world-to-image transformation
rotation_matrix = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta), np.cos(theta)]
])
pcd_rotated = pcd_relative @ rotation_matrix.T

# Step 3: Convert from meters to pixels
pcd_pixels = pcd_rotated / resolution

# Step 4: Apply image coordinate transformation
# Image y-axis points down, so flip y
x_indices = (pcd_pixels[:, 0] + W / 2 + dx_offset).astype(int)
y_indices = (H / 2 - pcd_pixels[:, 1] + dy_offset).astype(int)

# Clip to image bounds
x_indices = np.clip(x_indices, 0, W - 1)
y_indices = np.clip(y_indices, 0, H - 1)

# Transform vehicle heading for visualization
# Vehicle position relative to aerial center
vehicle_relative = np.array([T_current[0, 3] - aerial_easting, 
                             T_current[1, 3] - aerial_northing])
vehicle_rotated = vehicle_relative @ rotation_matrix.T
vehicle_pixel = vehicle_rotated / resolution
vehicle_x = int(vehicle_pixel[0] + W / 2 + dx_offset)
vehicle_y = int(H / 2 - vehicle_pixel[1] + dy_offset)

# Rotate heading to match aerial image orientation
heading_in_aerial_frame = heading_oxts - aerial_heading

